# pipe_nuevo — 02 FE: panel + shares + lags + features nuevas, TODO en DuckDB

Lee el parquet densificado por `01_Preprocesamiento.ipynb` y arma el panel de
features en DuckDB (agregacion, shares jerarquicos, lags/medias moviles vía
funciones de ventana SQL, deltas, indices, edad de producto vs. edad del par,
recencia de compra, peso acumulado, y el control de leakage) — motor columnar
que puede spillear a disco y evita ir y viniendo entre Python y el engine para
cada paso pesado.

Solo dos cosas se resuelven fuera de SQL, porque ahi son mas simples y no son
el cuello de botella: la matriz de vecinos por correlacion (chica, un producto
por columna, necesita pivot + correlacion pairwise-completa) y el escalado por
fila (que vive en `03_Escalado.ipynb`).

**No escala nada ac谩**: graba un parquet "sin escalar" (`features_sin_escalar_*`)
para que probar los 4 metodos de `PARAM['metodo_escalado']` en `03_Escalado.ipynb`
sea liviano — no hace falta rehacer agregacion/shares/lags/vecinos cada vez, que
es la parte pesada.


In [ ]:
import json, time
from pathlib import Path

import duckdb
import pandas as pd
import polars as pl


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    import os
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_PRE = BUCKET / "datasets_pre"     # lo escribe 01_Preprocesamiento
RUTA_FE = BUCKET / "datasets_fe"      # cache sin escalar Y salida final de 03_Escalado
RUTA_FE.mkdir(parents=True, exist_ok=True)

print(f"BUCKET      : {BUCKET}")
print(f"crudos      : {DIR_RAW}")
print(f"preprocesado: {DIR_PRE}")
print(f"salida FE   : {RUTA_FE}")


In [ ]:
PARAM = {
    # ══ Deben coincidir con 01_Preprocesamiento (para leer el parquet correcto) ══
    'granularidad': 'pc',              # 'pc' -> cliente-producto | 'p' -> producto
    'solo_productos_target': True,

    # ══ Target ═══════════════════════════════════════════════════════════
    'horizonte': 2,

    # ══ Lags / medias moviles de la propia serie ════════════════════════
    'max_lags': 12,
    'ventanas_ma': (3, 6, 12),

    # ══ Shares jerarquicos ═══════════════════════════════════════════════
    'niveles_share': ('cat1', 'cat2', 'cat3', 'mercado'),
    'lags_share': 3,
    'techo_indice': 10.0,

    # ══ Vecinos por correlacion temporal ═════════════════════════════════
    # Corte fijo para definir quien es vecino de quien SIN mirar el futuro.
    # Coincide con el fin de 'meses_train' de pipe/03_Optuna.ipynb (201701-201905).
    'mes_corte': 201905,
    'n_vecinos': 3,

    'semilla': 102191,
}

G  = PARAM['granularidad']
H  = PARAM['horizonte']
L  = PARAM['max_lags']
ES_PC = G == 'pc'
KEYS  = ['product_id', 'customer_id'] if ES_PC else ['product_id']
KEYS_SQL = ", ".join(KEYS)

_grp = 'grpClienteProducto' if ES_PC else 'grpProducto'
_tgt = '_tgtFilter' if PARAM['solo_productos_target'] else ''
NOMBRE_PRE = f"sellin_zeroes_{_grp}{_tgt}.parquet"
NOMBRE_SIN_ESCALAR = (f"features_sin_escalar_{_grp}{_tgt}_{L}lags_share_{H}h"
                      f"_vec{PARAM['n_vecinos']}.parquet")

print(f"granularidad : {G}   (claves: {KEYS})")
print(f"leo preprocesado : {NOMBRE_PRE}")
print(f"corte vecinos    : {PARAM['mes_corte']}   n_vecinos={PARAM['n_vecinos']}")
print(f"salida (cache)   : {NOMBRE_SIN_ESCALAR}")


In [ ]:
t0 = time.time()
con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE TABLE panel_raw AS
    SELECT * FROM read_parquet('{DIR_PRE / NOMBRE_PRE}')
""")
con.execute(f"""
    CREATE OR REPLACE TABLE prod AS
    SELECT * FROM read_csv('{DIR_RAW / "tb_productos.txt"}', delim='\t', header=true)
""")

n_raw = con.sql("SELECT COUNT(*) FROM panel_raw").fetchone()[0]
n_prod = con.sql("SELECT COUNT(DISTINCT product_id) FROM panel_raw").fetchone()[0]
print(f"preprocesado: {n_raw:,} filas · {n_prod} productos"
      + (f" · {con.sql('SELECT COUNT(DISTINCT customer_id) FROM panel_raw').fetchone()[0]} clientes" if ES_PC else ""))
print(f"[{time.time()-t0:.0f}s]")


### Agregacion a la granularidad elegida + calendario continuo `m`


In [ ]:
t0 = time.time()

con.execute(f"""
    CREATE OR REPLACE TABLE panel_0 AS
    SELECT {KEYS_SQL}, periodo,
           SUM(tn) AS tn,
           SUM(cust_request_tn) AS req_tn,
           SUM(cust_request_qty) AS req_qty,
           MAX(plan_precios_cuidados) AS precios_cuidados,
           ((periodo // 100) * 12 + (periodo % 100)) AS m
    FROM panel_raw
    GROUP BY {KEYS_SQL}, periodo
""")

n0 = con.sql("SELECT COUNT(*) FROM panel_0").fetchone()[0]
rango = con.sql("SELECT MIN(periodo), MAX(periodo) FROM panel_0").fetchone()
print(f"panel agregado: {n0:,} filas · rango {rango[0]} -> {rango[1]}")
print(f"[{time.time()-t0:.0f}s]")


### Vida de la clave, categorias, y edad CAUSAL del par cliente-producto


In [ ]:
t0 = time.time()

# Ya viene densificado desde 01_Preprocesamiento: alcanza con MIN/MAX de 'm' por
# ventana (no hace falta una grilla como en pipe_2).
con.execute(f"""
    CREATE OR REPLACE TABLE panel_1 AS
    SELECT p.*,
           MIN(m) OVER (PARTITION BY {KEYS_SQL}) AS m_nace,
           MAX(m) OVER (PARTITION BY {KEYS_SQL}) AS m_muere,
           pr.cat1, pr.cat2, pr.cat3, pr.brand, pr.sku_size
    FROM panel_0 p
    LEFT JOIN prod pr USING (product_id)
""")
con.execute("""
    CREATE OR REPLACE TABLE panel_1 AS
    SELECT *,
           CASE WHEN m >= m_nace THEN m - m_nace ELSE -1 END AS edad_cliente_producto
    FROM panel_1
""")

n_ceros = con.sql("SELECT COUNT(*) FROM panel_1 WHERE tn = 0").fetchone()[0]
print(f"panel con vida y categorias: {n0:,} filas")
print(f"ceros de tn: {n_ceros:,} ({100*n_ceros/n0:.0f}%)")
print(f"[{time.time()-t0:.0f}s]")


### Totales producto / cliente / jerarquia (para shares) + edad de producto (NUEVO)


In [ ]:
t0 = time.time()

con.execute("""
    CREATE OR REPLACE TABLE tot_prod AS
    SELECT product_id, m, SUM(tn) AS tn_prod, COUNT(*) AS n_clientes_prod
    FROM panel_1 GROUP BY product_id, m
""")
if ES_PC:
    con.execute("""
        CREATE OR REPLACE TABLE tot_cli AS
        SELECT customer_id, m, SUM(tn) AS tn_cli, COUNT(*) AS n_productos_cli
        FROM panel_1 GROUP BY customer_id, m
    """)

# m_nace del PRODUCTO (primera venta a CUALQUIER cliente), no del par. Se calcula
# igual con granularidad 'p' (ahi coincide con edad_cliente_producto: menos ramas
# especiales aguas abajo).
con.execute("""
    CREATE OR REPLACE TABLE vida_prod AS
    SELECT product_id, MIN(m) AS m_nace_prod FROM tot_prod GROUP BY product_id
""")
con.execute("""
    CREATE OR REPLACE TABLE panel_1 AS
    SELECT p.*,
           CASE WHEN p.m >= v.m_nace_prod THEN p.m - v.m_nace_prod ELSE -1 END AS edad_producto
    FROM panel_1 p LEFT JOIN vida_prod v USING (product_id)
""")

for niv in PARAM['niveles_share']:
    if niv == 'mercado':
        con.execute("""
            CREATE OR REPLACE TABLE tot_mercado AS
            SELECT m, SUM(tn_prod) AS tn_mercado FROM tot_prod GROUP BY m
        """)
    else:
        con.execute(f"""
            CREATE OR REPLACE TABLE tot_{niv} AS
            SELECT pr.{niv} AS {niv}, tp.m, SUM(tp.tn_prod) AS tn_{niv}, COUNT(*) AS n_prod_{niv}
            FROM tot_prod tp LEFT JOIN prod pr USING (product_id)
            GROUP BY pr.{niv}, tp.m
        """)

print(f"tot_prod: {con.sql('SELECT COUNT(*) FROM tot_prod').fetchone()[0]:,} filas")
print(f"[{time.time()-t0:.0f}s]")


### Shares (que tan importante es este producto/cliente en cada nivel)


In [ ]:
t0 = time.time()

if ES_PC:
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT p.*, tp.tn_prod, tp.n_clientes_prod, tc.tn_cli, tc.n_productos_cli,
               CASE WHEN ABS(tc.tn_cli) > 1e-9 THEN p.tn / tc.tn_cli ELSE 0.0 END AS sh_prod_en_cli,
               CASE WHEN ABS(tp.tn_prod) > 1e-9 THEN p.tn / tp.tn_prod ELSE 0.0 END AS sh_cli_en_prod
        FROM panel_1 p
        LEFT JOIN tot_prod tp USING (product_id, m)
        LEFT JOIN tot_cli tc USING (customer_id, m)
    """)
    SHARES = ["sh_prod_en_cli", "sh_cli_en_prod"]
else:
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT p.*, tp.tn_prod, tp.n_clientes_prod
        FROM panel_1 p LEFT JOIN tot_prod tp USING (product_id, m)
    """)
    SHARES = []

for niv in PARAM['niveles_share']:
    if niv == 'mercado':
        con.execute("""
            CREATE OR REPLACE TABLE df AS
            SELECT d.*, tm.tn_mercado,
                   CASE WHEN ABS(tm.tn_mercado) > 1e-9 THEN d.tn_prod / tm.tn_mercado ELSE 0.0 END AS sh_prod_en_mercado
            FROM df d LEFT JOIN tot_mercado tm USING (m)
        """)
        SHARES.append("sh_prod_en_mercado")
    else:
        con.execute(f"""
            CREATE OR REPLACE TABLE df AS
            SELECT d.*, t.tn_{niv},
                   CASE WHEN ABS(t.tn_{niv}) > 1e-9 THEN d.tn_prod / t.tn_{niv} ELSE 0.0 END AS sh_prod_en_{niv}
            FROM df d LEFT JOIN tot_{niv} t ON d.{niv} = t.{niv} AND d.m = t.m
        """)
        SHARES.append(f"sh_prod_en_{niv}")

print(f"{len(SHARES)} shares: {SHARES}")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
# ── Chequeo: los shares del mismo denominador suman 1 por grupo (sobre TODO
# el dataset, no solo un mes -- barato porque es una agregacion en duckdb) ──
if ES_PC:
    mn, mx = con.sql("""
        SELECT MIN(s), MAX(s) FROM (
            SELECT customer_id, m, SUM(sh_prod_en_cli) AS s FROM df GROUP BY 1, 2
        )
    """).fetchone()
    print(f"suma sh_prod_en_cli por cliente-mes: min {mn:.4f}  max {mx:.4f}  (max deberia ser 1)")
    assert abs(mx - 1.0) < 1e-6, "sh_prod_en_cli no suma 1 en algun cliente-mes"

if 'cat3' in PARAM['niveles_share']:
    mn3, mx3 = con.sql("""
        SELECT MIN(s), MAX(s) FROM (
            SELECT cat3, m, SUM(sh_prod_en_cat3) AS s
            FROM (SELECT DISTINCT product_id, cat3, m, sh_prod_en_cat3 FROM df)
            GROUP BY 1, 2
        )
    """).fetchone()
    print(f"suma sh_prod_en_cat3 por cat3-mes  : min {mn3:.4f}  max {mx3:.4f}  (deberia ser 1)")
    assert abs(mx3 - 1.0) < 1e-6, "sh_prod_en_cat3 no suma 1 en algun cat3-mes"

print("chequeo de shares OK")


### Lags, medias moviles y pico historico (funciones de ventana SQL)


In [ ]:
t0 = time.time()

lag_exprs = [f"LAG(tn, {k}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS tn_lag{k}"
             for k in range(1, L + 1)]
ma_exprs = [f"AVG(tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
            f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS tn_ma{w}"
            for w in PARAM['ventanas_ma']]
qty_ma_exprs = [f"AVG(req_qty) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
                f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS qty_ma{w}"
                for w in PARAM['ventanas_ma'][:2]]
otros = [
    f"LAG(req_qty, 1) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS qty_lag1",
    f"AVG(req_tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
    f"ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS reqtn_ma3",
    f"MAX(tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
    f"ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS tn_pico_hasta_aca",
]
con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(lag_exprs + ma_exprs + qty_ma_exprs + otros)} FROM df")

print(f"lags 1..{L} + medias moviles {PARAM['ventanas_ma']} agregados")
print(f"[{time.time()-t0:.0f}s]")


### Deltas de share e indices (ratio recortado contra el mes anterior)


In [ ]:
t0 = time.time()

delta_exprs = []
for s in SHARES:
    delta_exprs += [f"LAG({s}, {k}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS {s}_lag{k}"
                    for k in range(1, PARAM['lags_share'] + 1)]
    delta_exprs += [f"AVG({s}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
                    f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS {s}_ma{w}"
                    for w in (3, 6)]
con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(delta_exprs)} FROM df")

d_exprs = []
for s in SHARES:
    d_exprs.append(f"({s} - {s}_lag1) AS {s}_d1")
    d_exprs.append(f"({s} - {s}_ma3) AS {s}_dma3")
    if PARAM['lags_share'] >= 3:
        d_exprs.append(f"({s} - {s}_lag3) AS {s}_d3")
con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(d_exprs)} FROM df")

TECHO = PARAM['techo_indice']
idx_exprs = [
    f"CASE WHEN ABS(tn_lag1) > 1e-9 THEN LEAST(GREATEST(tn / tn_lag1, 0.0), {TECHO}) ELSE NULL END AS idx_tn_mom",
    f"CASE WHEN ABS(tn_ma3) > 1e-9 THEN LEAST(GREATEST(tn / tn_ma3, 0.0), {TECHO}) ELSE NULL END AS idx_tn_vs_ma3",
    f"CASE WHEN ABS(tn_pico_hasta_aca) > 1e-9 THEN LEAST(GREATEST(tn / tn_pico_hasta_aca, 0.0), {TECHO}) ELSE NULL END AS idx_tn_vs_pico",
    f"CASE WHEN ABS(qty_lag1) > 1e-9 THEN LEAST(GREATEST(req_qty / qty_lag1, 0.0), {TECHO}) ELSE NULL END AS idx_qty_mom",
]
con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(idx_exprs)} FROM df")

print(f"deltas de share + 4 indices agregados. [{time.time()-t0:.0f}s]")


### Actividad reciente y recencia de compra (NUEVO)

`meses_sin_compra`: causal, `null` mientras la serie nunca vendio. Se calcula con `LAST_VALUE(... IGNORE NULLS)` (forward-fill nativo de DuckDB) excluyendo el mes actual del propio frame de la ventana.


In [ ]:
t0 = time.time()

con.execute("CREATE OR REPLACE TABLE df AS SELECT *, CAST(tn > 0 AS TINYINT) AS vendio FROM df")
con.execute(f"""
    CREATE OR REPLACE TABLE df AS
    SELECT *,
           AVG(CAST(vendio AS DOUBLE)) OVER (PARTITION BY {KEYS_SQL} ORDER BY m
               ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS frac_meses_con_venta_6,
           SUM(vendio) OVER (PARTITION BY {KEYS_SQL} ORDER BY m
               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS meses_con_venta_3,
           (periodo % 100) AS mes_del_anio,
           CAST(edad_cliente_producto BETWEEN 0 AND 6 AS TINYINT) AS es_nuevo
    FROM df
""")

con.execute(f"""
    CREATE OR REPLACE TABLE df AS
    SELECT *,
           LAST_VALUE(CASE WHEN tn > 0 THEN m END IGNORE NULLS) OVER (
               PARTITION BY {KEYS_SQL} ORDER BY m
               ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
           ) AS _ultimo_m_con_venta_prev
    FROM df
""")
con.execute("CREATE OR REPLACE TABLE df AS SELECT * EXCLUDE (_ultimo_m_con_venta_prev), "
            "(m - _ultimo_m_con_venta_prev) AS meses_sin_compra FROM df")

n_nulos = con.sql("SELECT COUNT(*) FROM df WHERE meses_sin_compra IS NULL").fetchone()[0]
n_tot = con.sql("SELECT COUNT(*) FROM df").fetchone()[0]
print(f"meses_sin_compra: {n_nulos:,} nulos ({100*n_nulos/n_tot:.0f}%, series que nunca "
      f"compraron a esa altura)")
print(f"[{time.time()-t0:.0f}s]")


### Peso acumulado de cliente y producto en las ventas totales (NUEVO)

Causal: usa solo ventas acumuladas HASTA el mes `m`, nunca meses futuros.


In [ ]:
t0 = time.time()

con.execute("""
    CREATE OR REPLACE TABLE tn_total_mes AS
    SELECT m, SUM(tn) AS tn_total_mes,
           SUM(SUM(tn)) OVER (ORDER BY m ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS tn_total_acum
    FROM panel_1 GROUP BY m
""")

con.execute("""
    CREATE OR REPLACE TABLE tot_prod_acum AS
    SELECT tp.product_id, tp.m,
           CASE WHEN ABS(tm.tn_total_acum) > 1e-9
                THEN SUM(tp.tn_prod) OVER (PARTITION BY tp.product_id ORDER BY tp.m
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / tm.tn_total_acum
                ELSE 0.0 END AS peso_producto_acum
    FROM tot_prod tp LEFT JOIN tn_total_mes tm USING (m)
""")
con.execute("""
    CREATE OR REPLACE TABLE df AS
    SELECT d.*, tpa.peso_producto_acum
    FROM df d LEFT JOIN tot_prod_acum tpa USING (product_id, m)
""")

if ES_PC:
    con.execute("""
        CREATE OR REPLACE TABLE tot_cli_acum AS
        SELECT tc.customer_id, tc.m,
               CASE WHEN ABS(tm.tn_total_acum) > 1e-9
                    THEN SUM(tc.tn_cli) OVER (PARTITION BY tc.customer_id ORDER BY tc.m
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / tm.tn_total_acum
                    ELSE 0.0 END AS peso_cliente_acum
        FROM tot_cli tc LEFT JOIN tn_total_mes tm USING (m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*, tca.peso_cliente_acum
        FROM df d LEFT JOIN tot_cli_acum tca USING (customer_id, m)
    """)

print("peso_producto_acum" + (" y peso_cliente_acum" if ES_PC else "") + f" agregados. [{time.time()-t0:.0f}s]")


### Vecinos por correlacion temporal: sustitutos y complementarios (NUEVO)

Unica parte que sale de DuckDB: la matriz producto x producto es chica
(un producto por columna) y necesita correlacion pairwise-completa (ignora
NaN par a par), que pandas resuelve mejor que una funcion de ventana SQL. Se
arma con corte fijo `mes_corte` (mismo patron que `nat_exp/02_DTW_clusters.ipynb`)
para no dejar que la relacion de vecindad dependa de meses de validacion/test;
una vez fijada la relacion, el resultado vuelve a DuckDB para el join.


In [ ]:
t0 = time.time()

M_CORTE = (PARAM['mes_corte'] // 100) * 12 + (PARAM['mes_corte'] % 100)
N_VEC = PARAM['n_vecinos']

wide = (con.sql(f"""
            SELECT product_id, m, tn_prod FROM tot_prod WHERE m < {M_CORTE}
        """).pl()
           .pivot(on="product_id", index="m", values="tn_prod")
           .sort("m")
           .drop("m"))
corr = wide.to_pandas().corr()   # product x product, Pearson, pairwise-completa

vecinos_rows = []
for p in corr.columns:
    s = corr[p].drop(labels=[p]).dropna()
    if s.empty:
        continue
    for rank, (vecino, r) in enumerate(s.sort_values().head(N_VEC).items(), start=1):
        vecinos_rows.append({"product_id": p, "tipo": "sustituto", "rank": rank,
                              "vecino_id": vecino, "corr": float(r)})
    for rank, (vecino, r) in enumerate(s.sort_values(ascending=False).head(N_VEC).items(), start=1):
        vecinos_rows.append({"product_id": p, "tipo": "complementario", "rank": rank,
                              "vecino_id": vecino, "corr": float(r)})

vecinos_df = pd.DataFrame(vecinos_rows)
con.register("vecinos_pl", vecinos_df)
con.execute("""
    CREATE OR REPLACE TABLE vecinos AS
    SELECT CAST(product_id AS BIGINT) AS product_id, tipo, rank,
           CAST(vecino_id AS BIGINT) AS vecino_id, corr
    FROM vecinos_pl
""")
con.unregister("vecinos_pl")

n_prod_vec = con.sql("SELECT COUNT(DISTINCT product_id) FROM vecinos").fetchone()[0]
print(f"vecinos calculados para {n_prod_vec} productos "
      f"(corte {PARAM['mes_corte']}, {N_VEC} sustitutos + {N_VEC} complementarios c/u)")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()

con.execute("""
    CREATE OR REPLACE TABLE feat_vecinos AS
    SELECT v.product_id, v.tipo, tp.m, AVG(tp.tn_prod) AS tn_vecino_prom
    FROM vecinos v
    LEFT JOIN tot_prod tp ON tp.product_id = v.vecino_id
    GROUP BY v.product_id, v.tipo, tp.m
""")
con.execute("""
    CREATE OR REPLACE TABLE feat_vecinos_piv AS
    SELECT product_id, m,
           MAX(CASE WHEN tipo = 'sustituto' THEN tn_vecino_prom END) AS tn_sustitutos_prom,
           MAX(CASE WHEN tipo = 'complementario' THEN tn_vecino_prom END) AS tn_complementarios_prom
    FROM feat_vecinos GROUP BY product_id, m
""")
con.execute("""
    CREATE OR REPLACE TABLE df AS
    SELECT d.*,
           COALESCE(fv.tn_sustitutos_prom, 0.0) AS tn_sustitutos_prom,
           COALESCE(fv.tn_complementarios_prom, 0.0) AS tn_complementarios_prom
    FROM df d LEFT JOIN feat_vecinos_piv fv USING (product_id, m)
""")

# se graba la lista de vecinos por producto para poder auditarla
_vecinos_export = {}
for pid, tipo, rank, vecino_id, r in con.sql(
        "SELECT product_id, tipo, rank, vecino_id, corr FROM vecinos ORDER BY product_id, tipo, rank"
    ).fetchall():
    d = _vecinos_export.setdefault(str(pid), {"sustitutos": [], "complementarios": []})
    clave = "sustitutos" if tipo == "sustituto" else "complementarios"
    d[clave].append(vecino_id)

with open(RUTA_FE / NOMBRE_SIN_ESCALAR.replace(".parquet", "_vecinos.json"), "w", encoding="utf-8") as f:
    json.dump(_vecinos_export, f, indent=2, ensure_ascii=False)

print(f"tn_sustitutos_prom / tn_complementarios_prom agregados. [{time.time()-t0:.0f}s]")


### Target (se arma DESPUES de tener todos los lags)


In [ ]:
t0 = time.time()

con.execute(f"""
    CREATE OR REPLACE TABLE df AS
    SELECT *,
           LEAD(tn, {H}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS clase_tn,
           (((m + {H} - 1) // 12) * 100) + ((m + {H} - 1) % 12) + 1 AS periodo_objetivo
    FROM df
""")

_sup = con.sql("SELECT COUNT(*) FROM df WHERE clase_tn IS NOT NULL").fetchone()[0]
_tot = con.sql("SELECT COUNT(*) FROM df").fetchone()[0]
print(f"filas con target      : {_sup:,}")
print(f"filas de inferencia   : {_tot - _sup:,}")
print(f"[{time.time()-t0:.0f}s]")


### Control de data leakage

Mismos 5 chequeos que `pipe_2/01_Preproceso_y_FE.ipynb`, mas uno nuevo sobre `mes_corte`. El chequeo de correlacion (#3) corre sobre TODO el dataset con `CORR()` nativo de DuckDB en vez de una muestra de 300k filas en NumPy: mismo chequeo, sin el paso intermedio de traer una muestra a Python.


In [ ]:
errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


print("CONTROL DE DATA LEAKAGE")
print("=" * 74)

# ── 1) El shift del target apunta donde debe ─────────────────────────────
_una = con.sql(f"""
    SELECT {KEYS_SQL} FROM df WHERE clase_tn IS NOT NULL
    GROUP BY {KEYS_SQL} ORDER BY COUNT(*) DESC LIMIT 1
""").fetchone()
_k = dict(zip(KEYS, _una))
_where = " AND ".join(f"{c} = {v}" for c, v in _k.items())
_serie = con.sql(f"SELECT m, tn, clase_tn FROM df WHERE {_where} ORDER BY m").pl()
_tn, _cl = _serie["tn"].to_list(), _serie["clase_tn"].to_list()
_malos = [i for i in range(len(_tn) - H)
          if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
chk(not _malos, f"clase_tn[i] == tn[i+{H}] en la serie {_k} ({len(_tn)} meses, {len(_malos)} discrepancias)")

# ── 2) Ninguna columna del futuro entre las features ──────────────────────
info = con.sql("DESCRIBE df").fetchall()
tipos = {r[0]: r[1].upper().split("(")[0] for r in info}
PROHIBIDAS = {"clase_tn", "periodo_objetivo", "m_muere", "m", "periodo", "m_nace"} | set(KEYS)
FEATURES = [c for c in tipos if c not in PROHIBIDAS]
chk(not (set(FEATURES) & {"clase_tn", "periodo_objetivo"}), "el target no esta entre las features")
chk("m_muere" not in FEATURES, "m_muere (ultimo mes con venta = dato del futuro) NO es feature")
chk("m_nace" not in FEATURES, "m_nace no es feature; la edad causal la reemplaza")

# ── 3) Ninguna feature es el target disfrazado (CORR nativo, sobre TODO el dataset) ──
NUMERIC = {"TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT", "UTINYINT",
           "USMALLINT", "UINTEGER", "UBIGINT", "FLOAT", "DOUBLE", "DECIMAL"}
NUM_FEATURES = [c for c in FEATURES if tipos[c] in NUMERIC]
_sel = ", ".join(f'CORR(clase_tn, "{c}") AS "{c}"' for c in NUM_FEATURES)
_res = con.sql(f"SELECT {_sel} FROM df WHERE clase_tn IS NOT NULL").fetchone()
_sosp = [(c, round(v, 5)) for c, v in zip(NUM_FEATURES, _res) if v is not None and abs(v) > 0.999]
chk(not _sosp, f"ninguna de las {len(NUM_FEATURES)} features numericas correlaciona >0.999 "
               f"con clase_tn (sobre las {_tot:,} filas)  {_sosp}")

# ── 4) Los promedios moviles no miran adelante ───────────────────────────
_s2 = con.sql(f"SELECT m, tn, tn_ma3 FROM df WHERE {_where} ORDER BY m").pl()
_tn2, _ma = _s2["tn"].to_list(), _s2["tn_ma3"].to_list()
_err_ma = max((abs(_ma[i] - sum(_tn2[i-2:i+1]) / 3)
               for i in range(2, len(_tn2)) if _ma[i] is not None), default=0.0)
chk(_err_ma < 1e-9, f"tn_ma3[t] == promedio(tn[t-2..t]): error maximo {_err_ma:.2e}")

# ── 5) Los shares no usan el futuro: el denominador es del mismo mes ─────
if 'cat3' in PARAM['niveles_share']:
    _f = con.sql(f"SELECT m, cat3, tn_prod, sh_prod_en_cat3 FROM df WHERE {_where} AND tn > 0 LIMIT 1").fetchone()
    if _f:
        _mm, _c3, _tnprod, _obs = _f
        _tot_cat3 = con.sql(f"""
            SELECT SUM(tn_prod) FROM (
                SELECT DISTINCT product_id, tn_prod FROM df WHERE m = {_mm} AND cat3 = '{_c3}'
            )
        """).fetchone()[0]
        _esp = _tnprod / _tot_cat3 if _tot_cat3 else 0.0
        chk(abs(_esp - _obs) < 1e-6,
            f"sh_prod_en_cat3 recalculado a mano coincide (esperado {_esp:.6f}, observado {_obs:.6f})")

# ── 6) El corte de vecinos no mira meses de validacion/test ─────────────
chk(PARAM['mes_corte'] <= 201905,
    f"mes_corte ({PARAM['mes_corte']}) no supera el fin de 'meses_train' de "
    f"pipe/03_Optuna.ipynb (201905) -- si cambio ese split, actualizar mes_corte")

print("=" * 74)
if errores:
    raise RuntimeError(f"Control de leakage FALLIDO: {errores}")
print(f"Control superado. {len(FEATURES)} features ({len(NUM_FEATURES)} numericas).")


### Guardado del cache (COPY directo a parquet, sin pasar por Python)


In [ ]:
t0 = time.time()

DROP = {"m_nace", "m_muere"}
CTX_F64 = {"clase_tn", "tn0"}
exprs = []
for name, tipo, *_r in info:
    if name in DROP:
        continue
    tipo_base = tipo.upper().split("(")[0]
    out_name = "tn0" if name == "tn" else name
    if tipo_base == "DOUBLE" and out_name not in CTX_F64:
        exprs.append(f'CAST("{name}" AS FLOAT) AS "{out_name}"')
    else:
        exprs.append(f'"{name}" AS "{out_name}"')
select_sql = ",\n       ".join(exprs)

path_out = RUTA_FE / NOMBRE_SIN_ESCALAR
con.execute(f"""
    COPY (SELECT {select_sql} FROM df ORDER BY {KEYS_SQL}, m)
    TO '{path_out}' (FORMAT parquet)
""")

print(f"Guardado: {path_out}")
print(f"  {_tot:,} filas x {len(exprs)} columnas")
print(f"  tamanio en disco: {path_out.stat().st_size / 1e6:.0f} MB")
print(f"[{time.time()-t0:.0f}s]")
print()
print("listo para 03_Escalado.ipynb (metodo_escalado configurable sin recalcular esto)")
